# Large Models Test Baseline

Closed-book API baseline for larger reference models on all 60 test problems. The default list is Qwen-family first, with DeepSeek left as an optional upper-bound comparison.


## Setup

This notebook currently uses OpenRouter's OpenAI-compatible API. Put `OPENROUTER_API_KEY=...` in the repo-local `.env` file.

In [20]:
!pip install -q -U openai pandas tqdm python-dotenv

In [21]:
from pathlib import Path
import json
import os
import sys
import time

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

In [22]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
PROJECT_ROOT

PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune')

In [23]:
from training_eval.eval_utils import (
    GRADING_POLICY,
    default_test_dir,
    extract_answer,
    extract_json_object,
    is_correct,
    load_jsonl_records,
    make_closed_book_prompt,
    rows_to_frame,
    save_results,
    summarize_accuracy,
)

## Load Test Records


In [24]:
DATA_DIR = default_test_dir(PROJECT_ROOT)
records = load_jsonl_records(DATA_DIR, pattern="*_preview.jsonl")
len(records), DATA_DIR

(60,
 PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/benchmark/data/test'))

## Configure API Models

Now that the API has credit, the default references are larger Qwen-family models. Keep this list flexible: comment out anything slow, expensive, or rate-limited.


In [25]:
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

MODEL_NAMES = [
    "qwen/qwen3-8b",
    "qwen/qwen3-32b",
    "qwen/qwen3-235b-a22b",
    "deepseek/deepseek-v4-flash:free",
    "openai/gpt-oss-20b:free",
]

MAX_TOKENS = 128

RESET_OUTPUTS_BEFORE_RUN = True

RERUN_API_ERRORS = False
RERUN_OUTPUT_ERRORS = True
MAX_OUTPUT_RETRY_PASSES = 10
FILL_ONLY_EXISTING_EMPTY_OUTPUTS = False
STOP_MODEL_ON_RATE_LIMIT = True


In [26]:
def safe_model_name(model_name):
    return model_name.replace("-", "_").replace(".", "_").replace("/", "_").replace(":", "_")


def result_dir_for(model_name):
    return PROJECT_ROOT / "results" / "baselines" / f"{safe_model_name(model_name)}_test_closed_book_api"


def outputs_path_for(model_name):
    return result_dir_for(model_name) / "outputs.jsonl"


def load_cached_rows(model_name):
    path = outputs_path_for(model_name)
    if not path.exists():
        return []
    rows_by_id = {}
    with path.open() as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                rows_by_id[row["id"]] = row
    return list(rows_by_id.values())


def reset_model_outputs(model_names):
    for model_name in model_names:
        result_dir = result_dir_for(model_name)
        for filename in ["outputs.jsonl", "outputs.csv", "metrics.json"]:
            path = result_dir / filename
            if path.exists():
                path.unlink()


def append_cached_row(model_name, row):
    result_dir_for(model_name).mkdir(parents=True, exist_ok=True)
    with outputs_path_for(model_name).open("a") as f:
        f.write(json.dumps(row, sort_keys=True) + "\n")


def should_rerun_cached_row(row):
    if row.get("api_error"):
        return RERUN_API_ERRORS
    if row.get("output_error"):
        return RERUN_OUTPUT_ERRORS
    if RERUN_OUTPUT_ERRORS and not str(row.get("raw_output", "")).strip():
        return True
    if RERUN_OUTPUT_ERRORS and row.get("predicted_answer") is None:
        return True
    return False


def is_rate_limit_error(error_text):
    if not error_text:
        return False
    text = str(error_text).lower()
    return "429" in text or "rate limit" in text or "rate_limit" in text or "temporarily rate-limited" in text


def save_model_results(model_name, rows):
    df = rows_to_frame(rows)
    result_dir = result_dir_for(model_name)
    metrics = summarize_accuracy(df)
    metrics.update({
        "model": model_name,
        "provider": "openrouter",
        "dataset": "benchmark/data/test/*.jsonl",
        "num_api_errors": int(df["api_error"].notna().sum()) if "api_error" in df else 0,
        "num_output_errors": int(df["output_error"].notna().sum()) if "output_error" in df else 0,
        "num_empty_outputs": int(df["raw_output"].fillna("").astype(str).str.strip().eq("").sum()) if "raw_output" in df else 0,
        "checkpointed_after_each_record": True,
        "grading_policy": GRADING_POLICY,
    })
    return save_results(rows, result_dir, metrics)


def generate_answer(problem, model_name):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": make_closed_book_prompt(problem)}],
        temperature=0,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content or ""


def build_result_row(record, raw_output, predicted, api_error, output_error, correct):
    metadata = record.get("metadata", {})
    return {
        "id": record["id"],
        "family": record["family"],
        "problem_type": record["problem_type"],
        "difficulty": record["difficulty"],
        "manual_variation": metadata.get("manual_variation", False),
        "manual_problem_variation": metadata.get("manual_problem_variation", False),
        "manual_reasoning_variation": metadata.get("manual_reasoning_variation", False),
        "problem": record["problem"],
        "canonical_answer": record["canonical_answer"],
        "raw_output": raw_output,
        "predicted_answer": predicted,
        "api_error": api_error,
        "output_error": output_error,
        "correct": correct,
    }


def run_one_record(record, model_name):
    raw_output = ""
    predicted = None
    api_error = None
    output_error = None

    try:
        raw_output = generate_answer(record["problem"], model_name)
    except Exception as exc:
        api_error = repr(exc)
        return build_result_row(record, raw_output, predicted, api_error, output_error, False)

    if not raw_output.strip():
        output_error = "empty_response"
        return build_result_row(record, raw_output, predicted, api_error, output_error, False)

    predicted = extract_answer(raw_output, record["canonical_answer"])
    if predicted is None:
        output_error = "parse_failure"
        return build_result_row(record, raw_output, predicted, api_error, output_error, False)

    correct = is_correct(predicted, record["canonical_answer"])
    return build_result_row(record, raw_output, predicted, api_error, output_error, correct)


## One-Example Smoke Test

This runs the first dev record and writes it into the normal `outputs.jsonl` cache. The full run will skip it instead of spending the call again.


## Fresh Run Reset

When `RESET_OUTPUTS_BEFORE_RUN` is true, this clears cached API outputs before the smoke/full run. Use this for rigorous fresh baseline reruns.


In [27]:
if RESET_OUTPUTS_BEFORE_RUN:
    reset_model_outputs(MODEL_NAMES)


In [28]:
smoke_record = records[0]

for model_name in MODEL_NAMES:
    cached_rows = load_cached_rows(model_name)
    cached_by_id = {row["id"]: row for row in cached_rows if not should_rerun_cached_row(row)}

    print(model_name)
    if smoke_record["id"] in cached_by_id:
        row = cached_by_id[smoke_record["id"]]
        print("cached smoke result")
        print(row["raw_output"])
        print("predicted:", row["predicted_answer"])
        print("canonical:", row["canonical_answer"])
        print("correct:", row["correct"])
        print("api_error:", row.get("api_error"))
        print("output_error:", row.get("output_error"))
        print()
        continue

    row = run_one_record(smoke_record, model_name)
    append_cached_row(model_name, row)
    print(row["raw_output"])
    print("predicted:", row["predicted_answer"])
    print("canonical:", smoke_record["canonical_answer"])
    print("correct:", row["correct"])
    print("api_error:", row.get("api_error"))
    print("output_error:", row.get("output_error"))
    print()


qwen/qwen3-8b
<answer>{"expected_time": "21"}</answer>
predicted: {'expected_time': '21'}
canonical: {'expected_time': '21'}
correct: True
api_error: None
output_error: None

qwen/qwen3-32b
<answer>
{"expected_time": "21"}
</answer>
predicted: {'expected_time': '21'}
canonical: {'expected_time': '21'}
correct: True
api_error: None
output_error: None

qwen/qwen3-235b-a22b
<answer>
{"expected_time": "21"}
</answer>
predicted: {'expected_time': '21'}
canonical: {'expected_time': '21'}
correct: True
api_error: None
output_error: None

deepseek/deepseek-v4-flash:free

predicted: None
canonical: {'expected_time': '21'}
correct: False
api_error: RateLimitError("Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'deepseek/deepseek-v4-flash:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Crucible', 'is_byok': False}

## Run Full Test Evaluation

This cell is safe to use with **Run All** for a rigorous baseline.

It skips terminal cached rows, runs missing rows, and checkpoints each completed row immediately to `outputs.jsonl`. Non-API issues such as schema aliases are handled by the shared tolerant grader.

API failures are recorded with `api_error` and are not retried. Output failures that are not API failures, such as empty responses or unparsable JSON, are retried for up to `MAX_OUTPUT_RETRY_PASSES` passes. If only API-failed rows remain, the model is considered done for this run.


In [29]:
all_results = {}


def runnable_rows_for(model_name):
    cached_rows = load_cached_rows(model_name)
    terminal_rows = [row for row in cached_rows if not should_rerun_cached_row(row)]
    terminal_ids = {row["id"] for row in terminal_rows}
    unfinished = [record for record in records if record["id"] not in terminal_ids]
    return cached_rows, terminal_rows, terminal_ids, unfinished


for model_name in MODEL_NAMES:
    model_hit_rate_limit = False

    for pass_idx in range(1, MAX_OUTPUT_RETRY_PASSES + 1):
        cached_rows, rows, completed_ids, unfinished = runnable_rows_for(model_name)

        if not unfinished:
            print(f"{model_name}: complete ({len(completed_ids)}/{len(records)})")
            all_results[model_name] = rows
            break

        print(f"{model_name}: pass {pass_idx}/{MAX_OUTPUT_RETRY_PASSES}; {len(completed_ids)}/{len(records)} terminal rows, {len(unfinished)} rows to run/retry")

        for record in tqdm(unfinished):
            row = run_one_record(record, model_name)
            rows.append(row)
            append_cached_row(model_name, row)

            if row.get("api_error"):
                print(f"API issue for {model_name} on {record['id']}: {row['api_error']}")
            elif row.get("output_error"):
                print(f"Output issue for {model_name} on {record['id']}: {row['output_error']}")

            if STOP_MODEL_ON_RATE_LIMIT and is_rate_limit_error(row.get("api_error")):
                print(f"Giving up on {model_name} for this run after rate-limit error on {record['id']}")
                model_hit_rate_limit = True
                break

        rows = load_cached_rows(model_name)
        save_model_results(model_name, rows)
        all_results[model_name] = rows

        if model_hit_rate_limit:
            break

    else:
        cached_rows, rows, completed_ids, unfinished = runnable_rows_for(model_name)
        print(f"{model_name}: reached MAX_OUTPUT_RETRY_PASSES with {len(unfinished)} non-terminal rows still unresolved")
        all_results[model_name] = rows

print("Baseline run finished. API failures were saved and not retried; output failures were retried before giving up.")


qwen/qwen3-8b: pass 1/10; 1/60 terminal rows, 59 rows to run/retry


100%|██████████| 59/59 [34:51<00:00, 35.45s/it]   


qwen/qwen3-8b: complete (60/60)
qwen/qwen3-32b: pass 1/10; 1/60 terminal rows, 59 rows to run/retry


  2%|▏         | 1/59 [00:01<01:48,  1.87s/it]

Output issue for qwen/qwen3-32b on hitting_time_expectation_dev_004101: empty_response


...

Output issue for openai/gpt-oss-20b:free on stopped_process_expectation_dev_003100: empty_response


100%|██████████| 59/59 [14:29<00:00, 14.74s/it]


openai/gpt-oss-20b:free: pass 2/10; 58/60 terminal rows, 2 rows to run/retry


100%|██████████| 2/2 [01:19<00:00, 39.62s/it]

openai/gpt-oss-20b:free: complete (60/60)
Baseline run finished. API failures were saved and not retried; output failures were retried before giving up.


## Metrics

In [30]:
{model: len(rows) for model, rows in all_results.items()} if "all_results" in globals() else {}


{'qwen/qwen3-8b': 60,
 'qwen/qwen3-32b': 59,
 'qwen/qwen3-235b-a22b': 60,
 'deepseek/deepseek-v4-flash:free': 2,
 'openai/gpt-oss-20b:free': 60}

In [31]:
for model_name, rows in all_results.items():
    df = rows_to_frame(rows)
    print(model_name)
    print(f"Overall accuracy: {df['correct'].mean():.3f} ({df['correct'].sum()}/{len(df)})")
    display(df.groupby("family")["correct"].agg(["mean", "sum", "count"]).sort_index())
    display(df.groupby("difficulty")["correct"].agg(["mean", "sum", "count"]).sort_index())
    display(df.groupby("manual_variation")["correct"].agg(["mean", "sum", "count"]).sort_index())
    print()

qwen/qwen3-8b
Overall accuracy: 0.967 (58/60)


,mean,sum,count
family,,,
hitting_time_expectation,0.933333,14,15
martingale_verification,1.000000,15,15
optional_stopping_validity,1.000000,15,15
stopped_process_expectation,0.933333,14,15


,mean,sum,count
difficulty,,,
1,1.00,20,20
2,0.95,19,20
3,0.95,19,20


,mean,sum,count
manual_variation,,,
False,1.000000,24,24
True,0.944444,34,36



qwen/qwen3-32b
Overall accuracy: 0.831 (49/59)


...

,mean,sum,count
manual_variation,,,
True,0.0,0,2



openai/gpt-oss-20b:free
Overall accuracy: 0.900 (54/60)


,mean,sum,count
family,,,
hitting_time_expectation,0.866667,13,15
martingale_verification,1.000000,15,15
optional_stopping_validity,1.000000,15,15
stopped_process_expectation,0.733333,11,15


,mean,sum,count
difficulty,,,
1,0.95,19,20
2,0.85,17,20
3,0.90,18,20


,mean,sum,count
manual_variation,,,
False,0.875000,21,24
True,0.916667,33,36


## Save Results

In [32]:
saved_paths = {}

for model_name, rows in all_results.items():
    saved_paths[model_name] = save_model_results(model_name, rows)

saved_paths


{'qwen/qwen3-8b': (PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen_qwen3_8b_test_closed_book_api/outputs.jsonl'),
  PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen_qwen3_8b_test_closed_book_api/metrics.json'),
  PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen_qwen3_8b_test_closed_book_api/outputs.csv')),
 'qwen/qwen3-32b': (PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen_qwen3_32b_test_closed_book_api/outputs.jsonl'),
  PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen_qwen3_32b_test_closed_book_api/metrics.json'),
  PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen_qwen3_32b_test_closed_book_api/outputs.csv')),
 'qwen/qwen3-235b-a22b': (PosixPath('/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune/results/baselines/qwen_qwen3_235

## Inspect Mistakes

In [33]:
available_models = [model_name for model_name, rows in all_results.items() if rows]
available_models


['qwen/qwen3-8b',
 'qwen/qwen3-32b',
 'qwen/qwen3-235b-a22b',
 'deepseek/deepseek-v4-flash:free',
 'openai/gpt-oss-20b:free']

In [34]:
model_name = available_models[-1]
df = rows_to_frame(all_results[model_name])
df.loc[~df["correct"], [
    "id",
    "family",
    "problem_type",
    "difficulty",
    "canonical_answer",
    "predicted_answer",
    "api_error",
    "raw_output",
]].head(20)


,id,family,problem_type,difficulty,canonical_answer,predicted_answer,api_error,raw_output
11,hitting_time_expectation_dev_004301,hitting_time_expectation,biased_boundaries_zero_a,3,{'expected_time': '22980/2059'},{'expected_time': '11.16'},None,"<answer>{""expected_time"":""11.16""}"
13,hitting_time_expectation_dev_004303,hitting_time_expectation,biased_boundaries_zero_a,3,{'expected_time': '1265/211'},{'expected_time': '6.0'},None,"<answer>{""expected_time"":""6.0""}</answer>"
45,stopped_process_expectation_dev_003100,stopped_process_expectation,bounded_walk_expectation,1,{'value': '2'},{'value': '2.0'},None,"<answer>{""value"":""2.0""}</answer>"
52,stopped_process_expectation_dev_003202,stopped_process_expectation,quadratic_fixed_horizon,2,{'value': '84'},{'value': '67'},None,"<answer>{""value"":""67""}</answer>"
53,stopped_process_expectation_dev_003203,stopped_process_expectation,quadratic_fixed_horizon,2,{'value': '11'},{'E[S_10^2]': 11},None,"<answer>{\n ""E[S_10^2]"": 11\n}</answer>"
54,stopped_process_expectation_dev_003204,stopped_process_expectation,quadratic_fixed_horizon,2,{'value': '72'},{'value': '63'},None,"<answer>{""value"":""63""}</answer>"
